# Alpha and beta diversity patterns

In this notebook, the diversity metrics were measured for the abundance feature table.

We first set up the notebook and fetch the files generated with Euler stored on a polybox drive.

In [ ]:
import os
import pandas as pd
import qiime2 as q2
from qiime2 import Visualization
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.cm as cm

from scipy import stats

%matplotlib inline
%load_ext rpy2.ipython

In [ ]:
data_dir = 'updog_data'
os.makedirs(data_dir, exist_ok=True)

In [ ]:
!wget -O "$data_dir/updog-feature-table.qza" "https://polybox.ethz.ch/index.php/s/3NwEzFbbXSxZYxX/download"

In [ ]:
!wget -O "$data_dir/updog_metadata.tsv" "https://polybox.ethz.ch/index.php/s/pna5PZy62SfGcq5/download"

## 1. Alpha Rarefaction

In order to establish the the sampling depth, we first proceed with the alpha rarefaction.

In [ ]:
! qiime diversity alpha-rarefaction \
    --i-table $data_dir/updog-feature-table.qza \
    --p-max-depth 10000 \
    --m-metadata-file $data_dir/updog_metadata.tsv \
    --o-visualization $data_dir/alpha-rarefaction.qzv

In [ ]:
Visualization.load(f"{data_dir}/alpha-rarefaction.qzv")

We then settled for a sampling depth of 2000.

## 2. Core metrics

We then run the core metrics command to obtain alpha and beta diversity measurements.

In [ ]:
! qiime diversity core-metrics \
  --i-table $data_dir/updog-feature-table.qza \
  --m-metadata-file $data_dir/updog_metadata.tsv \
  --p-sampling-depth 2000 \
  --output-dir $data_dir/core-metrics-results

## 3. Significance
We then run the significance commands to generate visualisations and p/q-values for the relevant diversity metrics.
### 3.1. Shannon's index

In [ ]:
! qiime diversity alpha-group-significance \
  --i-alpha-diversity $data_dir/core-metrics-results/shannon_vector.qza \
  --m-metadata-file $data_dir/updog_metadata.tsv \
  --o-visualization $data_dir/core-metrics-results/shannon-group-significance.qzv

### 3.2. Observed Features

In [ ]:
! qiime diversity alpha-group-significance \
  --i-alpha-diversity $data_dir/core-metrics-results/observed_features_vector.qza \
  --m-metadata-file $data_dir/updog_metadata.tsv \
  --o-visualization $data_dir/core-metrics-results/observed_features-group-significance.qzv

### 3.3. Bray-Curtis PERMANOVA for Lifestyle

In [ ]:
! qiime diversity beta-group-significance \
  --i-distance-matrix $data_dir/core-metrics-results/bray_curtis_distance_matrix.qza \
  --m-metadata-file $data_dir/updog_metadata.tsv \
  --m-metadata-column subsistence_mode \
  --o-visualization $data_dir/core-metrics-results/bray-curtis-lifestyle-significance.qzv \
  --p-pairwise

### 3.4. Bray-Curtis PERMANOVA for other covariables

In [ ]:
# By Country
! qiime diversity beta-group-significance \
  --i-distance-matrix $data_dir/core-metrics-results/bray_curtis_distance_matrix.qza \
  --m-metadata-file $data_dir/updog_metadata.tsv \
  --m-metadata-column country \
  --o-visualization $data_dir/core-metrics-results/bray-curtis-country-significance.qzv \
  --p-pairwise

# By Sex
! qiime diversity beta-group-significance \
  --i-distance-matrix $data_dir/core-metrics-results/bray_curtis_distance_matrix.qza \
  --m-metadata-file $data_dir/updog_metadata.tsv \
  --m-metadata-column sex \
  --o-visualization $data_dir/core-metrics-results/bray-curtis-sex-significance.qzv \
  --p-pairwise

### 3.5. Bray-Curtis Adonis PERMANOVA

We also run a Adonis PERMANOVA to measure the significance of the influence of all our variables on the bray-curtis distance. We first run the command counting for interaction factors.

In [ ]:
metadata = pd.read_csv(f'{data_dir}/updog_metadata.tsv', sep='\t', index_col=0)
metadata['bmi'] = metadata['bmi'].fillna(metadata['bmi'].mean())
metadata.to_csv(f'{data_dir}/metadata_numeric.tsv', sep='\t')

In [ ]:
! qiime diversity adonis \
    --i-distance-matrix $data_dir/core-metrics-results/bray_curtis_distance_matrix.qza \
    --m-metadata-file $data_dir/metadata_numeric.tsv \
    --p-formula "subsistence_mode*country*sex*age*bmi" \
    --o-visualization $data_dir/core-metrics-results/bray-curtis-adonis.qzv 
    

As there is no interaction factor that showed a significant impact, we run the command again, but this time without any variable interaction.

In [ ]:
! qiime diversity adonis \
    --i-distance-matrix $data_dir/core-metrics-results/bray_curtis_distance_matrix.qza \
    --m-metadata-file $data_dir/metadata_numeric.tsv \
    --p-formula "subsistence_mode + country + sex + age + bmi" \
    --o-visualization $data_dir/core-metrics-results/bray-curtis-adonis-simple.qzv 

### 3.6. Jaccard PERMANOVA for lifestyle

In [ ]:
! qiime diversity beta-group-significance \
  --i-distance-matrix $data_dir/core-metrics-results/jaccard_distance_matrix.qza \
  --m-metadata-file $data_dir/updog_metadata.tsv \
  --m-metadata-column subsistence_mode \
  --o-visualization $data_dir/core-metrics-results/jaccard-lifestyle-significance.qzv \
  --p-pairwise

## 4. Diversity Visualizations
In this section are the codes for the visualizations for our report, since the QIIME2 visualizations cannot be personalized as much.
### 4.1 Alpha diversity boxplot

In [ ]:
! qiime tools export \
    --input-path $data_dir/core-metrics-results/shannon_vector.qza \
    --output-path $data_dir/core-metrics-results/shannon_export

In [ ]:
df_shannon = pd.read_csv(f'{data_dir}/core-metrics-results/shannon_export/alpha-diversity.tsv', index_col=0, sep='\t')
df_shannon

In [ ]:
md = pd.read_csv(f'{data_dir}/updog_metadata.tsv', sep='\t', index_col=0)
data_shannon = df_shannon.join(md)
data_shannon.head()

In [ ]:
def add_sig_line(ax, x1, x2, y, text):
    ax.plot([x1, x1, x2, x2],
            [y, y+0.03, y+0.03, y],
            lw=1, c='black')
    ax.text((x1 + x2) * 0.5,
            y + 0.035,
            text,
            ha='center',
            va='bottom')


We want the boxplots to show the significance levels, so we indicated the significant values and have them showed as stars.

In [ ]:
groups = list(data_shannon['subsistence_mode'].unique())

# example
p_west_farmer = 0.000003
p_west_fisher = 0.028092
p_west_hungat = 0.000068

def p_to_stars(p):
    if p < 0.001:
        return '***'
    elif p < 0.01:
        return '**'
    elif p < 0.05:
        return '*'
    else:
        return 'ns'

y_max = data_shannon['shannon_entropy'].max()
y_step = 0.1 * (y_max - data_shannon['shannon_entropy'].min())

y_min = data_shannon['shannon_entropy'].min()
y_step2 = (data_shannon['shannon_entropy'].max() -
          data_shannon['shannon_entropy'].min()) * 0.05

baseline = y_min - y_step2      # first bar below the boxes

In [ ]:

# generate n colors spanning the full viridis range
palette = sns.color_palette("viridis", n_colors=4)

sns.set(rc={'figure.figsize': (7, 5)}, style='white')



ax = sns.boxplot(
    data=data_shannon,
    x='subsistence_mode',
    y='shannon_entropy',
    palette=palette, flierprops=dict(marker='o', markersize=5)
)

ax.set_xlabel('Lifestyle')
ax.set_ylabel('Shannon diversity')

add_sig_line(ax,
             groups.index('Western'),
             groups.index('Fisher'),
             y_max + y_step*1,
             p_to_stars(p_west_fisher))


add_sig_line(ax,
             groups.index('Western'),
             groups.index('Farmer'),
             y_max + y_step*2,
             p_to_stars(p_west_farmer))



add_sig_line(ax,
             groups.index('Western'),
             groups.index('HunGat'),
             y_max + y_step*3,
             p_to_stars(p_west_hungat))

plt.savefig("shannon_boxplot.png", dpi=300, bbox_inches='tight')

#### Shannon diversity per country

we repeat the same steps but this time comparing countries instead of lifestyle.

In [ ]:
groups = list(data_shannon['country'].unique())

# example
p_USA_Tan = 0.000020
p_USA_Cam = 0.000221
p_USA_Per = 0.001859
p_Tan_Cam = 0.018614
p_Tan_Ita = 0.001829
p_Tan_Per = 0.002673

def p_to_stars(p):
    if p < 0.001:
        return '***'
    elif p < 0.01:
        return '**'
    elif p < 0.05:
        return '*'
    else:
        return 'ns'

y_max = data_shannon['shannon_entropy'].max()
y_step = 0.1 * (y_max - data_shannon['shannon_entropy'].min())

In [ ]:
# generate n colors spanning the full viridis range
palette = sns.color_palette("plasma", n_colors=5)

sns.set(rc={'figure.figsize': (7, 5)}, style='white')



ax = sns.boxplot(
    data=data_shannon,
    x='country',
    y='shannon_entropy', 
    palette=palette, flierprops=dict(marker='o', markersize=5)
)

ax.set_xlabel('Country')
ax.set_ylabel('Shannon diversity')

add_sig_line(ax,
             groups.index('USA'),
             groups.index('Tanzania'),
             y_max + y_step*5,
             p_to_stars(p_USA_Tan))


add_sig_line(ax,
             groups.index('USA'),
             groups.index('Cameroon'),
             y_max + y_step*1,
             p_to_stars(p_USA_Cam))


add_sig_line(ax,
             groups.index('USA'),
             groups.index('Peru'),
             y_max + y_step*3,
             p_to_stars(p_USA_Per))

add_sig_line(ax,
             groups.index('Tanzania'),
             groups.index('Cameroon'),
             y_max + y_step*2,
             p_to_stars(p_Tan_Cam))

add_sig_line(ax,
             groups.index('Tanzania'),
             groups.index('Italy'),
             y_max + y_step*6,
             p_to_stars(p_Tan_Ita))

add_sig_line(ax,
             groups.index('Tanzania'),
             groups.index('Peru'),
             y_max + y_step*4,
             p_to_stars(p_Tan_Per))

plt.savefig("shannon_country_boxplot.png", dpi=300, bbox_inches='tight')

### 4.2 PCoA plots
#### Bray-Curtis
We want to generate the Bray-Curtis PCoA plots for lifestyle, country, and sex, as all these variables had a significant impact on the distances, and some of the pairwise comparisons were also significantly different. 

In [ ]:
! qiime tools export \
    --input-path $data_dir/core-metrics-results/bray_curtis_pcoa_results.qza \
    --output-path $data_dir/core-metrics-results/pcoa_bray_curtis_export

In [ ]:

pcoa_bc = pd.read_csv(f'{data_dir}/core-metrics-results/pcoa_export/ordination.txt', 
    sep='\t', 
    skiprows=9, header=None,
    index_col=0
)


# keep first two axes
pcoa_bc = pcoa_bc.iloc[:, :2]
pcoa_bc.columns = ['PC1', 'PC2']

pcoa_bc = pcoa_bc.join(md)
pcoa_bc.head()


In [ ]:
fig, axes = plt.subplots(nrows=1, ncols=3, figsize=(24, 6), constrained_layout=True)

# Set Seaborn style and larger font scale
sns.set(style='white', rc={'figure.figsize': (24, 6)})
sns.set_context("notebook", font_scale=1.5)  # increase base font size

# Plot 1: subsistence_mode
sns.scatterplot(
    data=pcoa_bc,
    x='PC1',
    y='PC2',
    hue='subsistence_mode',
    palette='viridis',
    s=120, 
    style='subsistence_mode',
    alpha=0.8, ax=axes[0]
)
axes[0].set_xlabel('PC1 (9.3%)', fontsize=16)
axes[0].set_ylabel('PC2 (6.7%)', fontsize=16)
handles, labels = axes[0].get_legend_handles_labels()
axes[0].legend(handles=handles, labels=labels, loc='upper left', bbox_to_anchor=(1, 1), fontsize=14)

# Plot 2: country
marker_dict = {'Cameroon': 'o', 'Italy': 's', 'Peru': '^', 'Tanzania': 'D', 'USA': 'v'}
sns.scatterplot(
    data=pcoa_bc,
    x='PC1',
    y='PC2',
    hue='country',
    palette='plasma',
    s=120,
    style='country',
    markers=marker_dict,
    alpha=0.8, ax=axes[1]
)
axes[1].set_xlabel('PC1 (9.3%)', fontsize=16)
axes[1].set_ylabel('PC2 (6.7%)', fontsize=16)
handles, labels = axes[1].get_legend_handles_labels()
axes[1].legend(handles=handles, labels=labels, loc='upper left', bbox_to_anchor=(1, 1), fontsize=14)


# Plot 3: sex
marker_sex = {'F': '^', 'M': 'v'}
palette_sex = {'F': 'forestgreen', 'M': 'purple'}
sns.scatterplot(
    data=pcoa_bc,
    x='PC1',
    y='PC2',
    hue='sex',
    palette=palette_sex,
    s=120,
    style='sex',
    markers=marker_sex,
    alpha=0.8, ax=axes[2]
)
axes[2].set_xlabel('PC1 (9.3%)', fontsize=16)
axes[2].set_ylabel('PC2 (6.7%)', fontsize=16)
handles, labels = axes[2].get_legend_handles_labels()
axes[2].legend(handles=handles, labels=labels, loc='upper left', bbox_to_anchor=(1, 1), fontsize=14)


plt.savefig("pcoa.png", dpi=300, bbox_inches='tight')

#### Jaccard

We also generate a PCoA plot for the Jaccard distance, even if Bray-Curtis metric whuch also takes into account abundances.

In [ ]:
! qiime tools export \
    --input-path $data_dir/core-metrics-results/jaccard_pcoa_results.qza \
    --output-path $data_dir/core-metrics-results/pcoa_jaccard_export

In [ ]:
pcoa_j = pd.read_csv(f'{data_dir}/core-metrics-results/pcoa_jaccard_export/ordination.txt', 
    sep='\t', 
    skiprows=9, header=None,
    index_col=0
)


# keep first two axes
pcoa_j = pcoa_j.iloc[:, :2]
pcoa_j.columns = ['PC1', 'PC2']

pcoa_j = pcoa_j.join(md)
pcoa_j.head()

In [ ]:
sns.set(style='white', rc={'figure.figsize': (7, 5)})
ax = sns.scatterplot(
    data=pcoa_j,
    x='PC1',
    y='PC2',
    hue='subsistence_mode',  # replace with any metadata column
    palette='viridis',
    s=50
)

ax.set_xlabel('PC1 (4.6%)')
ax.set_ylabel('PC2 (3.1%)')